# 1. Data Transformation and Training

In [ ]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv('train.csv')
df = df.dropna(subset=["subrogation"])

In [ ]:
# Claim_date and Weekends
df['claim_date'] = pd.to_datetime(df['claim_date'], errors='coerce')
df['is_weekend_claim'] = df['claim_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Seasons
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    return 'Fall'
df['season'] = df['claim_date'].dt.month.apply(get_season)

# Encoder (training)
from sklearn.preprocessing import LabelEncoder
le_season = LabelEncoder()
df['season'] = le_season.fit_transform(df['season'])
joblib.dump(le_season, "season_label_encoder.pkl")

# Liability features
df['liab_prct'] = pd.to_numeric(df['liab_prct'], errors='coerce').fillna(0)
df['liability_minus_risk'] = df['liab_prct'] - df['safety_rating']

df['provability_score'] = (
    (df['witness_present_ind'] == 'Y').astype(int).fillna(0)
    + df['policy_report_filed_ind'].fillna(0)
    + df['email_or_tel_available'].fillna(0)
    + (df['in_network_bodyshop'] == 'yes').astype(int).fillna(0))

df['claim_est_payout'] = df['claim_est_payout'].fillna(0)

# Liability Binning
bins = list(range(20, 60, 4))
labels = [f"{b}-{b+4}" for b in bins[:-1]]
df['liab_bin'] = pd.cut(df['liab_prct'], bins=bins, labels=labels, include_lowest=True)
df['liab_bin'] = df['liab_bin'].astype(str).fillna("Other")

# Above Liability Threshold
df['above_liability_threshold'] = (df['liab_prct'] > 50).astype(int)

# Log Transformations
df['annual_income'] = np.log1p(df['annual_income'])

df['claim_est_payout'] = np.log1p(df['claim_est_payout'])

# Claims binning
df['claims_binned'] = pd.cut(
    df['past_num_of_claims'],
    bins=[-1, 0, 3, float('inf')],
    labels=['None', 'Medium', 'High']).astype('category')

# Data preprocessing
df['witness_flag'] = (df['witness_present_ind'] == 'Y').astype(int)

df['accident_type'] = df['accident_type'].astype(str)
df['accident_site'] = df['accident_site'].astype(str)

# liability x accident type
for t in df['accident_type'].unique():
    safe_t = t.replace('/', '_')
    col = f"liab_if_type_{safe_t}"
    df[col] = df['liab_prct'] * (df['accident_type'] == t).astype(int)

# liability x accident site
for s in df['accident_site'].unique():
    safe_s = s.replace('/', '_')
    col = f"liab_if_site_{safe_s}"
    df[col] = df['liab_prct'] * (df['accident_site'] == s).astype(int)

# liability x witness
df['liab_if_witness']    = df['liab_prct'] * df['witness_flag']
df['liab_if_nowitness']  = df['liab_prct'] * (1 - df['witness_flag'])

# witness x accident type
for t in df['accident_type'].unique():
    safe_t = t.replace('/', '_')
    col = f"witness_if_type_{safe_t}"
    df[col] = df['witness_flag'] * (df['accident_type'] == t).astype(int)

# witness x accident site
for s in df['accident_site'].unique():
    safe_s = s.replace('/', '_')
    col = f"witness_if_site_{safe_s}"
    df[col] = df['witness_flag'] * (df['accident_site'] == s).astype(int)

# accident type x accident site
for t in df['accident_type'].unique():
    for s in df['accident_site'].unique():
        safe_t = t.replace('/', '_')
        safe_s = s.replace('/', '_')
        col = f"type_{safe_t}__site_{safe_s}"
        df[col] = ((df['accident_type'] == t).astype(int) *
                   (df['accident_site'] == s).astype(int))

# liability x witness x accident type x accident site
df['liab_type_site_witness'] = (
    df['liab_prct'] *
    df['witness_flag'] *
    df['accident_type'].astype('category').cat.codes *
    df['accident_site'].astype('category').cat.codes
)

# nonlinear features
df['liab_prct_sq'] = df['liab_prct'] ** 2
df['liab_prct_cu'] = df['liab_prct'] ** 3

# recoverable amounts
df['recoverable_amount'] = ((100 - df['liab_prct']) / 100) * df['claim_est_payout']
df['recoverable_amount_log'] = np.log1p(df['recoverable_amount'])

# liability distance from threshold at 50
df['liab_dist_50'] = df['liab_prct'] - 50
df['liab_dist_abs_50'] = df['liab_dist_50'].abs()

# saving training means (for testing / prediction)
if 'subrogation' in df.columns:
    mean_liab_sub = df[df['subrogation'] == 1]['liab_prct'].mean()
    mean_liab_non = df[df['subrogation'] == 0]['liab_prct'].mean()
    joblib.dump({'sub': mean_liab_sub, 'non': mean_liab_non}, "liab_means.pkl")

# submeans
df['liab_dist_submean'] = df['liab_prct'] - mean_liab_sub
df['liab_dist_nonmean'] = df['liab_prct'] - mean_liab_non

# claims velocity
df['claim_velocity'] = (
        (df['past_num_of_claims'] == 0) * 0.0 +
        (df['past_num_of_claims'].between(1, 3)) * 0.5 +
        (df['past_num_of_claims'] >= 4) * 1.0
)

df['claim_velocity_scaled'] = df['claim_velocity'] * df['liab_prct'] / 100.0

# remove columns
df = df.drop(columns=['claim_number', 'zip_code', 'recoverable_amount'],
             errors='ignore')

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

# train, test split
train_data, test_data = train_test_split(df, test_size=0.2, stratify=df['subrogation'], random_state=2025)

# # # # # # # # # # # #
# Training Autogluon
predictor = TabularPredictor(
    label='subrogation',
    eval_metric='f1',
    problem_type='binary'
)
predictor.fit(
    train_data=train_data,
    presets='best',
    time_limit=1200 * 2, # 40 minutes
    num_stack_levels=1,
    hyperparameter_tune_kwargs={"num_trials": 20, "scheduler": "local", "searcher": "auto"}
)

# Save model
predictor.save("autogluon_model")

In [ ]:
# # # # # # # # # # # #
# Testing Autogluon
path =  "PATH/TO/MODEL"

predictor = TabularPredictor.load(path)

# Evaluate performance
performance = predictor.evaluate(test_data)
performance

{'f1': 0.5795574288724974,
 'accuracy': 0.7783333333333333,
 'balanced_accuracy': np.float64(0.7396169104749086),
 'mcc': 0.4397444774305262,
 'roc_auc': np.float64(0.8316233284080173),
 'precision': 0.5116279069767442,
 'recall': 0.6682867557715675}

In [ ]:
# Get feature importance using test data
fi = predictor.feature_importance(test_data)
fi

These features in provided data are not utilized by the predictor and will be ignored: ['witness_flag']
Computing feature importance via permutation shuffling for 71 features using 3600 rows with 5 shuffle sets...
	1365.04s	= Expected runtime (273.01s per shuffle set)
	729.67s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
accident_type,0.018345,0.006075,0.001254,5,0.030854,0.005836
liab_if_type_single_car,0.014486,0.006283,0.003359,5,0.027422,0.001550
email_or_tel_available,0.010407,0.003312,0.001081,5,0.017226,0.003587
accident_site,0.010192,0.004580,0.003809,5,0.019622,0.000762
high_education_ind,0.008345,0.006559,0.023319,5,0.021851,-0.005161
...,...,...,...,...,...,...
type_multi_vehicle_clear__site_Highway_Intersection,-0.002934,0.001507,0.993938,5,0.000169,-0.006037
claim_date,-0.003203,0.002772,0.969450,5,0.002505,-0.008911
gender,-0.004099,0.005060,0.927854,5,0.006319,-0.014517
provability_score,-0.004941,0.001103,0.999720,5,-0.002669,-0.007212
